In [2]:
import helpers
from helpers import init_batch, generate_next_token
from helpers import merge_batches, filter_batch

helpers reload


ValueError: Tokenizer class Qwen2Tokenizer does not exist or is not currently imported.

In [ ]:
import copy
import random
import matplotlib.pyplot as plt
import torch
import time
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
model_name = "G:/model_weights/models/model/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [4]:
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

tokenizer.pad_size = "left"
tokenizer.truncation_size = "left"

In [5]:
prompts = [
    "The quick brown fox jumped over the",
    "The rain in Spain falls",
    "What comes up must",
]

# note: padding=True ensures the padding token will be inserted into the tokenized tensors
inputs = tokenizer(prompts, padding=True, return_tensors="pt")

In [6]:
def generate_batch_tokens_with_past(inputs):
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    last_logits = logits[:, -1, :]
    next_token_ids = last_logits.argmax(dim=1, keepdim=True)
    return next_token_ids, outputs.past_key_values


def generate_batch(inputs, max_tokens):
    # create a list of tokens for every input in the batch
    generated_tokens = [[] for _ in range(inputs["input_ids"].shape[0])]
    
    attention_mask = inputs["attention_mask"]
    position_ids = attention_mask.long().cumsum(-1) - 1
    position_ids.masked_fill_(attention_mask == 0, 1)
    
    next_inputs = {
        "position_ids": position_ids,
        **inputs
    }
    for _ in range(max_tokens):
        next_token_ids, past_key_values = generate_batch_tokens_with_past(next_inputs)
        next_inputs = {
            "input_ids": next_token_ids,  # '-1' here means the remaining elements for this dim
            "position_ids": next_inputs["position_ids"][:, -1].unsqueeze(-1) + 1,  # increment last, discard the rest
            "attention_mask": torch.cat([
                next_inputs["attention_mask"],
                torch.ones((next_token_ids.shape[0], 1)),  # concatenate vector of 1's with shape [batch_size]
            ], dim=1),
            "past_key_values": past_key_values,
        }

        next_tokens = tokenizer.batch_decode(next_token_ids)
        for i, token in enumerate(next_tokens):
            generated_tokens[i].append(token)
    return ["".join(tokens) for tokens in generated_tokens]

In [74]:
random.seed(42)
queue_size = 32
batch_size = 8

request_queue = [
    (prompts[0], 100 if i % batch_size == 0 else 10)
    for i in range(queue_size)
]

In [75]:
batches = [
    request_queue[i: i + batch_size]
    for i in range(0, len(request_queue), batch_size)
]
batches[:2]

[[('The quick brown fox jumped over the', 100),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10)],
 [('The quick brown fox jumped over the', 100),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10)]]

In [ ]:
from transformers import DynamicCache
def merge_batches(batch1, batch2):
    # first find the max sequence length of the two batches
    # this can be obtained from the second dimension 
    # of the attention mask
    attn_mask1 = batch1["attention_mask"]
    attn_mask2 = batch2["attention_mask"]
    max_seq_len = max(attn_mask1.shape[1], attn_mask2.shape[1])
    
    # pad each mask (on the left) to the max sequence length
    # attention mask uses 0 for padding
    padding1 = max_seq_len - attn_mask1.shape[1]
    padding2 = max_seq_len - attn_mask2.shape[1]
    attn_mask1 = F.pad(attn_mask1, (padding1, 0), "constant", 0)
    attn_mask2 = F.pad(attn_mask2, (padding2, 0), "constant", 0)
    
    # because we only append batches post decoding, 
    # we don't need to pad input_ids
    # or position_ids. these are always length 1 
    # in the sequence dimension
    # however, we do need to pad the 
    # past_key_values, which have shape:
    # [batch_size, num_heads, sequence_length, head_dim]
    past_kv1 = batch1["past_key_values"]
    past_kv2 = batch2["past_key_values"]
    
    key_cache_1 = []
    value_cache_1 = []
    for i, (k, v, _) in enumerate(past_kv1):
        k = F.pad(k, (0, 0, padding1, 0), "constant", 0)
        v = F.pad(v, (0, 0, padding1, 0), "constant", 0)     
        key_cache_1.append(k)
        value_cache_1.append(v)
    past_kv1.key_cache = key_cache_1
    past_kv1.value_cache = value_cache_1
    
    key_cache_2 = []
    value_cache_2 = []
    # k, v shape [batch_size, num_head, seq_len, head_dim]
    for k, v, _ in past_kv2:
        k = F.pad(k, (0, 0, padding2, 0), "constant", 0)
        v = F.pad(v, (0, 0, padding2, 0), "constant", 0)     
        key_cache_2.append(k)
        value_cache_2.append(v)
    past_kv2.key_cache = key_cache_2
    past_kv2.value_cache = value_cache_2
        
    # now that everything has been padded to have
    # consistent shapes, let's merge
    input_ids = torch.concat(
        [batch1["input_ids"], batch2["input_ids"]], dim=0)
    position_ids = torch.concat(
        [batch1["position_ids"], batch2["position_ids"]], dim=0) 
    attn_mask = torch.concat([attn_mask1, attn_mask2], dim=0)
    
    past_key_value = []
    for i in range(len(key_cache_1)):
        k1, v1 = key_cache_1[i], value_cache_1[i]
        k2, v2 = key_cache_2[i], value_cache_2[i]
        k = torch.concat([k1, k2], dim=0)
        v = torch.concat([v1, v2], dim=0)
        past_key_value.append((k, v))
    
    return {
        "input_ids": input_ids,
        "position_ids": position_ids,
        "attention_mask": attn_mask,
        "past_key_values": past_kv1,
        "responses": batch1["responses"] + batch2["responses"],
        "tokens_remaining": batch1["tokens_remaining"] + batch2["tokens_remaining"],
    }

In [ ]:
t0 = time.time()
with tqdm(total=len(batches), desc=f"bs={batch_size}") as pbar:
    for i, batch in enumerate(batches):
        batch_max_tokens = [b[1] for b in batch]
        max_tokens = max(batch_max_tokens)
        pbar.set_postfix({"max_tokens": max_tokens})

        batch_prompts = [b[0] for b in batch]
        inputs = tokenizer(
            batch_prompts, padding=True, return_tensors="pt"
        )

        generate_batch(inputs, max_tokens=max_tokens)
        pbar.update(1)

duration_s = time.time() - t0
print("duration_s:", duration_s)

In [77]:
t0 = time.time()
with tqdm(total=len(batches), desc=f"bs={batch_size}") as pbar:
    print(request_queue)
    print(batch_size)
    batch = init_batch(request_queue[:batch_size])
    cached_batch = generate_next_token(batch)
    request_queue = request_queue[batch_size:]

    while (
        len(request_queue) > 0 or 
        cached_batch["input_ids"].size(0) > 0
    ):
        batch_capacity = (
            batch_size - cached_batch["input_ids"].size(0)
        )
        if batch_capacity > 0 and len(request_queue) > 0:
            # prefill
            new_batch = init_batch(request_queue[:batch_capacity])
            new_batch = generate_next_token(new_batch)
            request_queue = request_queue[batch_capacity:]

            # merge
            print(cached_batch["input_ids"].size())
            print(new_batch["input_ids"].size())
            cached_batch = merge_batches(cached_batch, new_batch)

        cached_batch = generate_next_token(cached_batch)
        cached_batch, removed_indices = filter_batch(cached_batch)
        pbar.update(1)
duration_s = time.time() - t0
print(duration_s)

bs=8:   0%|          | 0/4 [00:00<?, ?it/s]

[('The quick brown fox jumped over the', 100), ('The quick brown fox jumped over the', 10), ('The quick brown fox jumped over the', 10), ('The quick brown fox jumped over the', 10), ('The quick brown fox jumped over the', 10), ('The quick brown fox jumped over the', 10), ('The quick brown fox jumped over the', 10), ('The quick brown fox jumped over the', 10), ('The quick brown fox jumped over the', 100), ('The quick brown fox jumped over the', 10), ('The quick brown fox jumped over the', 10), ('The quick brown fox jumped over the', 10), ('The quick brown fox jumped over the', 10), ('The quick brown fox jumped over the', 10), ('The quick brown fox jumped over the', 10), ('The quick brown fox jumped over the', 10), ('The quick brown fox jumped over the', 100), ('The quick brown fox jumped over the', 10), ('The quick brown fox jumped over the', 10), ('The quick brown fox jumped over the', 10), ('The quick brown fox jumped over the', 10), ('The quick brown fox jumped over the', 10), ('The 

bs=8: 9it [00:02,  3.99it/s]                       

torch.Size([1, 1])
torch.Size([7, 1])


AttributeError: 'list' object has no attribute 'get_seq_length'

In [ ]:
a = torch.tensor([
    [1,2, 5], [3,4,5,]
])
print(a.size())

torch.Size([2, 3])


In [36]:
class itt:
    def __init__(self, a) -> None:
        pass
        self.a = a
    
    def __iter__(self):
        for i in self.a:
            yield i

a = itt([1,2,3])
b = itt([4,5,6])
# for i in a:
#     print(i)
for i in zip(a, b):
    print(i)
    

(1, 4)
(2, 5)
(3, 6)
